<a href="https://colab.research.google.com/github/SujahathMSM/Pytorch-DeepLearning/blob/main/BuildingLLMsFromScratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Data Sampling with Sliding Windows Approach**

In [3]:
!!pip install tiktoken

['Requirement already satisfied: tiktoken in /usr/local/lib/python3.12/dist-packages (0.12.0)',
 'Requirement already satisfied: regex>=2022.1.18 in /usr/local/lib/python3.12/dist-packages (from tiktoken) (2025.11.3)',
 'Requirement already satisfied: requests>=2.26.0 in /usr/local/lib/python3.12/dist-packages (from tiktoken) (2.32.4)',
 'Requirement already satisfied: charset_normalizer<4,>=2 in /usr/local/lib/python3.12/dist-packages (from requests>=2.26.0->tiktoken) (3.4.7)',
 'Requirement already satisfied: idna<4,>=2.5 in /usr/local/lib/python3.12/dist-packages (from requests>=2.26.0->tiktoken) (3.13)',
 'Requirement already satisfied: urllib3<3,>=1.21.1 in /usr/local/lib/python3.12/dist-packages (from requests>=2.26.0->tiktoken) (2.5.0)',
 'Requirement already satisfied: certifi>=2017.4.17 in /usr/local/lib/python3.12/dist-packages (from requests>=2.26.0->tiktoken) (2026.4.22)']

In [4]:
with open ("/content/the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

In [5]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')

In [6]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [7]:
enc_sample = enc_text[50:]

In [8]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287]


In [9]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(context, "----->", desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


In [10]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(tokenizer.decode(context), "----->", tokenizer.decode([desired]))

 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


In [11]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

# You need this class defined!
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.tokenizer = tokenizer
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt)

        # Create sliding windows
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [12]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [13]:
with open ("/content/the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=4,
    stride=1,
    shuffle=False
)

dataiter = iter(dataloader)

first_batch = next(dataiter)
print("first batch----", first_batch)

second_batch = next(dataiter)
print("second batch----", second_batch)

third_batch = next(dataiter)
print("third batch",third_batch)

first batch---- [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
second batch---- [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]
third batch [tensor([[2885, 1464, 1807, 3619]]), tensor([[1464, 1807, 3619,  402]])]


In [14]:
with open ("/content/the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False
)

dataiter = iter(dataloader)

first_batch = next(dataiter)
print("first batch----", first_batch)

second_batch = next(dataiter)
print("second batch----", second_batch)

third_batch = next(dataiter)
print("third batch",third_batch)

first batch---- [tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]]), tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])]
second batch---- [tensor([[  287,   262,  6001,   286],
        [  465, 13476,    11,   339],
        [  550,  5710,   465, 12036],
        [   11,  6405,   257,  5527],
        [27075,    11,   290,  4920],
        [ 2241,   287,   257,  4489],
        [   64,   319,   262, 34686],
        [41976,    13,   357, 10915]]), tensor([[  262,  6001,   286,   465],
        [

In [15]:
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False
)

dataiter = iter(dataloader)

inputs_1, targets_1 = next(dataiter)
inputs_2, targets_2 = next(dataiter)

print("Inputs\n", inputs_1)
print("Targets\n", targets_1)

Inputs
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [16]:
vocab_size = 50237
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [17]:
token_embedding_layer

Embedding(50237, 256)

In [19]:
token_embedding_layer.weight

Parameter containing:
tensor([[ 0.9089, -0.0339, -1.9304,  ..., -0.1848,  1.2066,  2.2029],
        [-0.0995, -0.1017, -0.1626,  ..., -1.1590,  0.7341, -0.8244],
        [-0.2179, -1.0906,  0.6652,  ..., -0.1051, -0.1603,  0.6499],
        ...,
        [ 0.2600,  0.7823,  1.1862,  ...,  0.4456, -1.8938,  1.1921],
        [ 1.0638,  0.3852, -0.6282,  ..., -0.4473, -1.8948,  0.9534],
        [-0.5499, -0.5935,  0.1353,  ..., -1.1090, -0.6183,  1.0418]],
       requires_grad=True)

In [20]:
max_lenght = 4
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=max_lenght,
    stride=4,
    shuffle=False
)

In [21]:
dataloader

In [22]:
dataiter = iter(dataloader)
dataiter

In [24]:
next(dataiter)

[tensor([[   40,   367,  2885,  1464],
         [ 1807,  3619,   402,   271],
         [10899,  2138,   257,  7026],
         [15632,   438,  2016,   257],
         [  922,  5891,  1576,   438],
         [  568,   340,   373,   645],
         [ 1049,  5975,   284,   502],
         [  284,  3285,   326,    11]]),
 tensor([[  367,  2885,  1464,  1807],
         [ 3619,   402,   271, 10899],
         [ 2138,   257,  7026, 15632],
         [  438,  2016,   257,   922],
         [ 5891,  1576,   438,   568],
         [  340,   373,   645,  1049],
         [ 5975,   284,   502,   284],
         [ 3285,   326,    11,   287]])]

In [25]:
next(dataiter)

[tensor([[  287,   262,  6001,   286],
         [  465, 13476,    11,   339],
         [  550,  5710,   465, 12036],
         [   11,  6405,   257,  5527],
         [27075,    11,   290,  4920],
         [ 2241,   287,   257,  4489],
         [   64,   319,   262, 34686],
         [41976,    13,   357, 10915]]),
 tensor([[  262,  6001,   286,   465],
         [13476,    11,   339,   550],
         [ 5710,   465, 12036,    11],
         [ 6405,   257,  5527, 27075],
         [   11,   290,  4920,  2241],
         [  287,   257,  4489,    64],
         [  319,   262, 34686, 41976],
         [   13,   357, 10915,   314]])]

In [26]:
len(dataiter)

160

In [27]:
print(f"Total number of batches in the dataloader: {len(dataloader)}")

Total number of batches in the dataloader: 160


In [28]:
input, targets = next(dataiter)
print(input)
print(targets)

tensor([[  314,  2138,  1807,   340],
        [  561,   423,   587, 10598],
        [  393, 28537,  2014,   198],
        [  198,     1,   464,  6001],
        [  286,   465, 13476,     1],
        [  438,  5562,   373,   644],
        [  262,  1466,  1444,   340],
        [   13,   314,   460,  3285]])
tensor([[ 2138,  1807,   340,   561],
        [  423,   587, 10598,   393],
        [28537,  2014,   198,   198],
        [    1,   464,  6001,   286],
        [  465, 13476,     1,   438],
        [ 5562,   373,   644,   262],
        [ 1466,  1444,   340,    13],
        [  314,   460,  3285,  9074]])


In [30]:
print('--- Getting the first batch again ---')
new_dataiter = iter(dataloader) # Create a new iterator
inputs, targets = next(new_dataiter)

print("Inputs (first batch again)\n", inputs)
print("Targets (first batch again)\n", targets)

print('\n--- Now the new_dataiter is at the second batch ---')
second_batch_from_new_iterator_inputs, second_batch_from_new_iterator_targets = next(new_dataiter)
print("Inputs (second batch from new iterator)\n", second_batch_from_new_iterator_inputs)

--- Getting the first batch again ---
Inputs (first batch again)
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets (first batch again)
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])

--- Now the new_dataiter is at the second batch ---
Inputs (second batch from new iterator)
 tensor([[  287,   262,  6001,   286],
        [  465, 13476,    11,   339],
        [  550,  5710,   465, 12036],
        [   11,  6405,   257,  5527],
        [27075,    11,   290,  4920],
        [ 2

In [34]:
token_embeddings = token_embedding_layer(inputs)
print(f"Shape of inputs (token IDs): {inputs.shape}")
print(f"Shape of token_embeddings (embedding vectors): {token_embeddings.shape}")

Shape of inputs (token IDs): torch.Size([8, 4])
Shape of token_embeddings (embedding vectors): torch.Size([8, 4, 256])


In [33]:
print(f"Shape of inputs (token IDs): {inputs.shape}")
print(f"Shape of token_embeddings (embedding vectors): {token_embeddings.shape}")

Shape of inputs (token IDs): torch.Size([8, 4])
Shape of token_embeddings (embedding vectors): torch.Size([8, 4, 256])
